# كشكول التجارب: البحث الهجين باستخدام LangChain (المحاضرة 42)

أهلاً بيك! في المحاضرة دي، مكتبة `LangChain` بتقدملنا أداة جاهزة وسحرية اسمها `EnsembleRetriever`.

بدل ما نقعد نحسب المصفوفات بإيدينا زي ما عملنا في الدرس اللي فات، `LangChain` بتعمل كل ده في خطوتين:
1. بتعمل مسترجع سريع بالكلمات (`BM25Retriever`).
2. بتعمل مسترجع ذكي بالفيكتورز (`FAISS`).
3. وبتجمعهم مع بعض بوزن إنت بتحدده!

تعال نجربها خطوة بخطوة بطريقة سهلة.


In [1]:
# 1. نستورد الحاجات اللي محتاجينها
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers.bm25 import BM25Retriever

try:
    from langchain.retrievers.ensemble import EnsembleRetriever
except ImportError:
    from langchain_classic.retrievers.ensemble import EnsembleRetriever

print("كله تمام والمكتبات جاهزة!")


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_21380\3949789112.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


كله تمام والمكتبات جاهزة!


### 2. شوية وثائق نجرب عليها
هنكتب 5 وثائق خفيفة (عن LangChain والـ AI ومستند واحد عن برج إيفل عشان نشوف هيطلع في النتايج ولا لأ):


In [2]:
docs = [
    Document(page_content="LangChain helps build LLM applications"),
    Document(page_content="LangChain can be used to develop Agentic AI applications"),
    Document(page_content="LangChain has many types of retrievers"),
    Document(page_content="Pinecone is a vector database for semantic search"),
    Document(page_content="The Eiffel Tower is located in Paris")  # جملة للتشتيت
]

print(f"عندنا {len(docs)} مستندات جاهزين للبحث!")


عندنا 5 مستندات جاهزين للبحث!


### 3. نجهز المسترجعين: الكثيف (FAISS) والمتناثر (BM25)


In [3]:
# المسترجع الكثيف: معتمد على التضمينات الدلالية
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# المسترجع المتناثر: معتمد على الكلمات المفتاحية
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3

print("المسترجع الدلالي والمسترجع الحرفي جاهزين!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

المسترجع الدلالي والمسترجع الحرفي جاهزين!


### 4. تعال ندمجهم باستخدام EnsembleRetriever
هنا بنحدد الأوزان: 70% للبحث الدلالي و 30% للبحث الحرفي:


In [4]:
# دمج المسترجعين
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)

# تعال نسأل سؤال ونشوف إيه اللي هيرجع
my_question = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(my_question)

print(f"السؤال: '{my_question}'\n")
print("أفضل النتائج اللي رجعت:")
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.page_content}")


السؤال: 'How can I build an application using LLMs?'

أفضل النتائج اللي رجعت:
  1. LangChain helps build LLM applications
  2. LangChain can be used to develop Agentic AI applications
  3. Pinecone is a vector database for semantic search
  4. The Eiffel Tower is located in Paris


---
### مساحتك للتجربة الحرة
جرب في الخلية دي تسأل سؤال تاني خالص، أو غير الأوزان وخليها مثلاً `[0.2, 0.8]` وشوف هل النتائج هتتغير؟


In [5]:
# جرب هنا:
# test_query = "Paris tourism"
# test_results = hybrid_retriever.invoke(test_query)
# for d in test_results:
#     print(d.page_content)
